# 05 · Export to ONNX and round-trip via `onnxruntime`

Demonstrates that the hand-built ONNX graph reproduces predictions to numerical precision, and is consumable by any ONNX-aware runtime (including OMLT externally).


In [ ]:
import os
os.environ["JAX_ENABLE_X64"] = "1"

import jax
jax.config.update("jax_enable_x64", True)

import numpy as np
import onnxruntime as ort

from jax_ldt import (
    HyperplaneTreeRegressor,
    to_onnx,
    to_json,
    from_json,
    predict_tree,
)


## Train and export

In [ ]:
rng = np.random.default_rng(0)
X = rng.uniform(-2, 2, size=(300, 3))
y = X[:, 0] ** 2 - X[:, 1] * X[:, 2]
model = HyperplaneTreeRegressor(
    max_depth=4, max_bins=6, min_samples_leaf=12, max_weight=1, num_terms=2,
).fit(X, y)

import tempfile, os
tmp = tempfile.mkdtemp()
onnx_path = os.path.join(tmp, "tree.onnx")
to_onnx(model, onnx_path)
size_kb = os.path.getsize(onnx_path) / 1024
print(f"wrote {onnx_path}: {size_kb:.1f} KB")


## Round-trip through onnxruntime

In [ ]:
sess = ort.InferenceSession(onnx_path)
X_test = X[:50].astype(np.float64)
yh_jax = np.asarray(model.predict(X_test))
yh_onnx = sess.run(None, {"X": X_test})[0].reshape(-1)

max_diff = float(np.max(np.abs(yh_jax - yh_onnx)))
print(f"max |jax - onnx|: {max_diff:.2e}")
np.testing.assert_allclose(yh_jax, yh_onnx, atol=1e-9)
print("round-trip OK to 1e-9")


## JSON spec round-trip too

In [ ]:
json_path = os.path.join(tmp, "tree.json")
to_json(model, json_path)
tree = from_json(json_path)
yh_json = np.asarray(predict_tree(tree, X_test))
np.testing.assert_allclose(yh_json, yh_jax, atol=1e-12)
print("JSON round-trip OK to 1e-12")


## Two artifacts, one source of truth

* **JSON** is the canonical, human-readable format. Use it to inspect a trained tree, version-control it, or feed it to a custom solver.
* **ONNX** is the export-for-other-runtimes format. The same file can be loaded by `onnxruntime`, by OMLT inside a Pyomo workflow, or by ONNX-compatible deployment toolchains — *we* don't depend on any of those at runtime.
